Developing a text generation system using a GPT-style pretrained transformer model from Hugging Face.
The model will generate coherent and contextually relevant text based on a given prompt.


In [195]:
# ============================================================
# CELL 1: IMPORT LIBRARIES
# ============================================================

#core libraries
import os
import torch
import pandas as pd
from datetime import datetime

# NLP libraries
from datasets import load_dataset
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments
)

# Set device to GPU if available, otherwise use CPU
device = 0 if torch.cuda.is_available() else -1

print("Libraries imported successfully.")
print("Device:", "GPU" if torch.cuda.is_available() else "CPU")

Libraries imported successfully.
Device: GPU


In [196]:
# ============================================================
# CELL 2: LOAD TEXT GENERATION MODEL
# ============================================================

# A function to load a text generation model (default is GPT-2) using Hugging Face's pipeline. 
# The model is loaded onto the appropriate device (GPU or CPU).

def load_generator(model_name="gpt2"):
    generator = pipeline(
        task="text-generation",
        model=model_name,
        tokenizer=model_name,
        device=device
    )

    print(f"Model loaded successfully: {model_name}")
    return generator


gpt2_generator = load_generator("gpt2")

Model loaded successfully: gpt2


In [197]:
# ============================================================
# CELL 3: CLEAN TEXT GENERATION FUNCTION
# ============================================================

# A function to generate text based on a given prompt using the loaded generator. 
# It includes parameters for controlling the generation process, such as max_new_tokens, temperature, top_k, top_p, and repetition_penalty. 
# The function also checks for valid input prompts and returns
def generate_text(
    generator,
    prompt,
    max_new_tokens=80,
    temperature=0.7,
    top_k=40,
    top_p=0.9,
    repetition_penalty=1.25
):
    if not prompt or not prompt.strip():
        return "Please enter a valid prompt."

    output = generator(
        prompt,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        do_sample=True,
        repetition_penalty=repetition_penalty,
        no_repeat_ngram_size=3,
        num_return_sequences=1,
        return_full_text=False,
        pad_token_id=generator.tokenizer.eos_token_id
    )

    return output[0]["generated_text"].strip()

In [198]:
# ============================================================
# CELL 4: PROMPT TEMPLATE SYSTEM
# ============================================================

# A function to create different prompt templates based on the specified mode (chatbot, educational, creative, professional). 
# This allows for more structured and context-aware text generation based on the user's input and the desired output style.

def create_prompt(user_input, mode="chatbot"):
    if mode == "chatbot":
        return f"""Answer the following question clearly and professionally.

Question: {user_input}

Answer:"""

    elif mode == "educational":
        return f"""Explain the following topic in simple words.

Topic: {user_input}

Explanation:"""

    elif mode == "creative":
        return f"""Write a creative paragraph about this topic.

Topic: {user_input}

Paragraph:"""

    elif mode == "professional":
        return f"""Write a professional explanation about this topic.

Topic: {user_input}

Explanation:"""

    else:
        return user_input


#chatbot response function that takes user input, creates a prompt using the create_prompt function, 
# and generates a response using the generate_text function with the specified parameters for text generation.

def chatbot_response(user_question, generator=gpt2_generator):
    prompt = create_prompt(user_question, mode="chatbot")

    response = generate_text(
        generator=generator,
        prompt=prompt,
        max_new_tokens=80,
        temperature=0.6,
        top_k=40,
        top_p=0.9,
        repetition_penalty=1.3
    )

    return response

In [199]:
# ============================================================
# CELL 5: TEST GPT-2 TEXT GENERATION
# ============================================================

# A simple test to generate text using the GPT-2 model with a sample prompt.
prompt = "Artificial Intelligence will change the future because"

response = generate_text(
    generator=gpt2_generator,
    prompt=prompt,
    max_new_tokens=80
)

print(response)

of it.
The problem is, there are only a few things that can be done about AI in this age:


In [200]:
# ============================================================
# CELL 6: TEST CHATBOT RESPONSE
# ============================================================

question = "What is artificial intelligence?"

answer = chatbot_response(question)

print(answer)

Artificial Intelligence (AI) refers to any computer program that can process information, learn from it or understand what other people are saying about you without ever having had a chance of hearing them! The term "intelligent" was coined by Carl Sagan in his book Cosmos 2. AI has been used as an acronym for many different types including humans who have no idea how things work; computers which do not know


In [201]:
# ============================================================
# CELL 7: TEST MULTIPLE PROMPTS AND SAVE OUTPUTS
# ============================================================

test_prompts = [
    "What is artificial intelligence?",
    "What is machine learning?",
    "How can robots help doctors?",
    "Why is deep learning important?",
    "How can AI improve education?"
]

results = []

for prompt in test_prompts:
    answer = chatbot_response(prompt)

    results.append({
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "model": "gpt2",
        "prompt": prompt,
        "generated_text": answer
    })

results_df = pd.DataFrame(results)
results_df.to_csv("outputs/generated_outputs.csv", index=False)

results_df

,timestamp,model,prompt,generated_text
0,2026-05-23 00:07:39,gpt2,What is artificial intelligence?,Artificial Intelligence (AI) has been around f...
1,2026-05-23 00:07:43,gpt2,What is machine learning?,Machine Learning refers to computer programs t...
2,2026-05-23 00:07:47,gpt2,How can robots help doctors?,They have been designed to perform tasks that ...
3,2026-05-23 00:07:51,gpt2,Why is deep learning important?,Deep Learning has been a critical component of...
4,2026-05-23 00:07:56,gpt2,How can AI improve education?,We are trying to find a way of improving educa...


In [202]:
# ============================================================
# CELL 8: LOAD GPT-NEO MODEL FOR COMPARISON
# ============================================================

# Optional but part of bonus features.
# This model may take time to download.

gpt_neo_generator = load_generator("EleutherAI/gpt-neo-125M")

Model loaded successfully: EleutherAI/gpt-neo-125M


In [203]:
# ============================================================
# CELL 9: COMPARE GPT-2 VS GPT-NEO
# ============================================================

# A function to compare the text generation outputs of GPT-2 and GPT-Neo models for a given prompt. 
# It generates responses from both models and creates a DataFrame to store the results for comparison.
def compare_models(prompt):
    formatted_prompt = create_prompt(prompt, mode="chatbot")

    gpt2_output = generate_text(
        generator=gpt2_generator,
        prompt=formatted_prompt,
        max_new_tokens=80,
        temperature=0.6,
        repetition_penalty=1.3
    )

    gpt_neo_output = generate_text(
        generator=gpt_neo_generator,
        prompt=formatted_prompt,
        max_new_tokens=80,
        temperature=0.6,
        repetition_penalty=1.3
    )

    comparison_df = pd.DataFrame([
        {
            "model": "GPT-2",
            "prompt": prompt,
            "generated_text": gpt2_output
        },
        {
            "model": "GPT-Neo 125M",
            "prompt": prompt,
            "generated_text": gpt_neo_output
        }
    ])

    return comparison_df


comparison_df = compare_models("What is artificial intelligence?")
comparison_df.to_csv("outputs/model_comparison_outputs.csv", index=False)

comparison_df

,model,prompt,generated_text
0,GPT-2,What is artificial intelligence?,Artificial Intelligence (AI) refers to machine...
1,GPT-Neo 125M,What is artificial intelligence?,Artificial Intelligence is an artificial intel...


In [204]:
# ============================================================
# CELL 10: LOAD YOUR DOWNLOADED CHATBOT CSV DATASET
# ============================================================

#load the CSV dataset containing daily dialogues for chatbot training or analysis.
csv_path = "dataset/daily_dailogue.csv"

df = pd.read_csv(csv_path)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

df.head()

Dataset loaded successfully.
Shape: (60005, 2)


,context,response
0,Waiter !,"I'll be with you in a second . Uh ... Yes , m..."
1,"I'll be with you in a second . Uh ... Yes , m...",This is not what I asked for . I'm afraid .
2,This is not what I asked for . I'm afraid .,"Oh , I'm so sorry . May I ask what you ordere..."
3,"Oh , I'm so sorry . May I ask what you ordere...","Yes . What I ordered is roast beef , not roas..."
4,Why are you always staring at Melissa ? Do you...,Mind your own business .


In [205]:
# ============================================================
# CELL 11: CLEAN AND PREPARE DATASET FOR GPT-2 FINE-TUNING
# ============================================================

# Keep only required columns
df = df[["context", "response"]]

# Remove missing values
df = df.dropna()

# Convert columns to string
df["context"] = df["context"].astype(str).str.strip()
df["response"] = df["response"].astype(str).str.strip()

# Remove empty rows
df = df[(df["context"] != "") & (df["response"] != "")]

# Convert into chatbot training format
df["text"] = "User: " + df["context"] + "\nAI Assistant: " + df["response"]

# Save prepared file
df[["text"]].to_csv("dataset/chatbot_prepared.csv", index=False)

print("Dataset prepared successfully.")
print("Total training samples:", len(df))

df[["text"]].head()

Dataset prepared successfully.
Total training samples: 60005


,text
0,User: Waiter !\nAI Assistant: I'll be with you...
1,User: I'll be with you in a second . Uh ... Ye...
2,User: This is not what I asked for . I'm afrai...
3,"User: Oh , I'm so sorry . May I ask what you o..."
4,User: Why are you always staring at Melissa ? ...


In [206]:
# ============================================================
# CELL 12: LOAD PREPARED DATASET FOR FINE-TUNING
# ============================================================

# Load the prepared CSV dataset for fine-tuning the GPT-2 model. 
# This dataset contains the formatted dialogues that will be used to train the model to generate appropriate responses based on user inputs.

custom_dataset = load_dataset(
    "csv",
    data_files={
        "train": "dataset/chatbot_prepared.csv"
    }
)

custom_dataset



Generating train split: 60005 examples [00:00, 330444.66 examples/s]


DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 60005
    })
})

In [207]:
# ============================================================
# CELL 13: TOKENIZE DATASET
# ============================================================

# Load the GPT-2 tokenizer and model, and tokenize the dataset to prepare it for fine-tuning.
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# GPT-2 does not have a padding token by default
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

print("Tokenizer and model loaded successfully.")

# A function to tokenize the dataset using the GPT-2 tokenizer. 
# It truncates and pads the input text to a maximum length of 128 tokens, which
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

# Tokenize the dataset by applying the tokenize_function to each example in the dataset.
tokenized_dataset = custom_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print(tokenized_dataset)

Tokenizer and model loaded successfully.

































Map: 100%|██████████| 60005/60005 [00:04<00:00, 13125.02 examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 60005
    })
})


In [208]:
# ============================================================
# REDUCE DATASET SIZE FOR FAST TRAINING
# ============================================================

#since our dataset is large and our memory is limited, we will reduce the dataset size for faster training.
small_train_dataset = tokenized_dataset["train"].shuffle(seed=42).select(range(10000))

print("Original samples:", len(tokenized_dataset["train"]))
print("Training samples used:", len(small_train_dataset))

Original samples: 60005
Training samples used: 10000


In [ ]:
# ============================================================
# CELL 14: TRAINING SETUP
# ============================================================

# Set up the data collator for language modeling, training arguments, and the Trainer for fine-tuning the GPT-2 model on the tokenized dataset.
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Define training arguments for fine-tuning the GPT-2 model, including output directory, number of epochs, batch size, 
# learning rate, and other parameters to control the training process and model saving.

training_args = TrainingArguments(
    output_dir="models/gpt2_finetuned_checkpoints",
    overwrite_output_dir=True,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    save_steps=500,
    save_total_limit=1,
    logging_steps=50,
    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_steps=50,
    prediction_loss_only=True,
    report_to="none"
)

# Initialize the Trainer with the model, training arguments, tokenized dataset, and data collator for fine-tuning the GPT-2 model.
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,  # use " train_dataset = tokenized_dataset["train"]" for full dataset, if you have enough memory and time
    data_collator=data_collator
)

print("Trainer is ready.")

Trainer is ready.


In [210]:
# this is an optional cell to clean up memory before starting the fine-tuning process, especially if you are running on a GPU with limited memory.
import gc

del gpt_neo_generator
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Memory cleaned. Ready for training.")

  1%|          | 59/6001 [06:57<11:40:09,  7.07s/it]

Memory cleaned. Ready for training.


In [211]:
# ============================================================
# CELL 15: FINE-TUNE GPT-2
# ============================================================

# Start the fine-tuning process for the GPT-2 model using the Trainer. 
# This will train the model on the tokenized dataset and save checkpoints according to the specified training arguments.
trainer.train()

  2%|▏         | 50/2500 [01:01<41:50,  1.02s/it]  

{'loss': 3.6912, 'grad_norm': 20.05765151977539, 'learning_rate': 5e-05, 'epoch': 0.02}


  4%|▍         | 100/2500 [01:54<50:41,  1.27s/it] 

{'loss': 2.9082, 'grad_norm': 24.419620513916016, 'learning_rate': 4.89795918367347e-05, 'epoch': 0.04}


  6%|▌         | 150/2500 [02:58<46:51,  1.20s/it] 

...

100%|██████████| 2500/2500 [45:16<00:00,  1.04s/it]

{'loss': 2.5841, 'grad_norm': 5.552114009857178, 'learning_rate': 0.0, 'epoch': 1.0}



100%|██████████| 2500/2500 [45:22<00:00,  1.09s/it]

{'train_runtime': 2722.0149, 'train_samples_per_second': 3.674, 'train_steps_per_second': 0.918, 'train_loss': 2.748455014038086, 'epoch': 1.0}


TrainOutput(global_step=2500, training_loss=2.748455014038086, metrics={'train_runtime': 2722.0149, 'train_samples_per_second': 3.674, 'train_steps_per_second': 0.918, 'total_flos': 653230080000000.0, 'train_loss': 2.748455014038086, 'epoch': 1.0})

In [212]:
# ============================================================
# CELL 16: SAVE FINE-TUNED MODEL
# ============================================================

# After fine-tuning the GPT-2 model, save the fine-tuned model and tokenizer to a specified directory for later use in text generation tasks.
fine_tuned_model_path = "models/fine_tuned_gpt2"

# save the fine-tuned model and tokenizer to the specified directory for later use in text generation tasks.
trainer.save_model(fine_tuned_model_path)
tokenizer.save_pretrained(fine_tuned_model_path)

print(f"Fine-tuned model saved at: {fine_tuned_model_path}")

Fine-tuned model saved at: models/fine_tuned_gpt2


In [213]:
# ============================================================
# CELL 17: LOAD FINE-TUNED MODEL
# ============================================================

# Load the fine-tuned GPT-2 model and tokenizer from the specified directory, and create a
# text generation pipeline using the fine-tuned model for generating responses based on the new training.

fine_tuned_generator = pipeline(
    task="text-generation",
    model=fine_tuned_model_path,
    tokenizer=fine_tuned_model_path,
    device=device
)

print("Fine-tuned GPT-2 model loaded successfully.")

Fine-tuned GPT-2 model loaded successfully.


In [214]:
# ============================================================
# CELL 18: FINE-TUNED CHATBOT RESPONSE
# ============================================================

# A chatbot response function that takes user input, creates a prompt using the create_prompt function, 
# and generates a response using the generate_text function with the specified parameters for text generation,
def fine_tuned_chatbot_response(user_question):
    prompt = f"User: {user_question}\nAI Assistant:"

    response = generate_text(
        generator=fine_tuned_generator,
        prompt=prompt,
        max_new_tokens=80,
        temperature=0.7,
        top_k=40,
        top_p=0.9,
        repetition_penalty=1.2
    )

    return response


question = "What is life?"

answer = fine_tuned_chatbot_response(question)

print(answer)

It's a simple job. You are supposed to be happy, but you have no idea what it will take to do well in school or the workplace.It can't happen without experience and good work ethic... there must also exist some fundamental learning skills that people need for success as professionals who help others achieve their dreams of becoming a successful professional. If we were all working at different companies instead, our


In [215]:
# ============================================================
# CELL 19: COMPARE BASE GPT-2 VS FINE-TUNED GPT-2
# ============================================================

# A function to compare the text generation outputs of the base GPT-2 model and the fine-tuned GPT-2 model for a given prompt. 
# It generates responses from both models and creates a DataFrame to store the results for comparison.
def compare_base_and_finetuned(prompt):
    base_answer = chatbot_response(prompt, generator=gpt2_generator)
    fine_tuned_answer = fine_tuned_chatbot_response(prompt)

    df = pd.DataFrame([
        {
            "model": "Base GPT-2",
            "prompt": prompt,
            "generated_text": base_answer
        },
        {
            "model": "Fine-Tuned GPT-2",
            "prompt": prompt,
            "generated_text": fine_tuned_answer
        }
    ])

    return df


fine_tuned_comparison_df = compare_base_and_finetuned("What is artificial intelligence?")
fine_tuned_comparison_df.to_csv("outputs/fine_tuned_comparison_outputs.csv", index=False)

fine_tuned_comparison_df

,model,prompt,generated_text
0,Base GPT-2,What is artificial intelligence?,Artificial Intelligence (AI) refers to a set o...
1,Fine-Tuned GPT-2,What is artificial intelligence?,"Artificial Intelligence, the term I use to des..."


In [216]:
# ============================================================
# CELL 20: FINAL INTERACTIVE CHATBOT
# ============================================================

# A simple interactive loop to chat with the fine-tuned GPT-2 chatbot.
print("GPT-Style Fine-Tuned Chatbot")
print("Type 'exit' to stop.\n")

while True:
    user_input = input("You: ")

    if user_input.lower() == "exit":
        print("Chatbot stopped.")
        break

    answer = fine_tuned_chatbot_response(user_input)

    print("\nAI:")
    print(answer)
    print("-" * 80)

GPT-Style Fine-Tuned Chatbot
Type 'exit' to stop.


AI:
I'm a writer. And if you want to write anything about science, then it's the best way to go. It helps me out more and better. If something happens on my blog in which there are a lot of people who think that we should not write about it anymore... well at least I won't be the one writing about it again.So yeah! Life isn ’ t a
--------------------------------------------------------------------------------
Chatbot stopped.
